# 08 - LangChain 工具调用（Tool Calling）

## 学习目标
- 使用 @tool 装饰器从函数创建工具
- 使用 Pydantic BaseModel 定义结构化参数
- 使用 StructuredTool.from_function() 编程式创建工具
- 处理工具异常（ToolException）
- 将工具绑定到 ChatOpenAI 模型（bind_tools）
- 在 LangGraph 中使用 ToolNode 执行工具调用
- 理解工具调用解析流程

In [ ]:
# 安装必要依赖（如未安装请取消注释）
# !pip install langchain langchain-openai langgraph pydantic

## 1. @tool 装饰器 - 从函数创建工具

@tool 装饰器是创建工具的最简单方式。它自动从函数签名和文档字符串生成工具描述和参数模式。

In [ ]:
from langchain_core.tools import tool
from pydantic import BaseModel, Field
from typing import Optional, List
import math

# === 示例1: 最简单的工具 - 仅用文档字符串描述 ===
@tool
def calculator(expression: str) -> str:
    """
    执行数学计算。支持基本运算: +, -, *, /, ** (幂), sqrt (平方根)。
    
    Args:
        expression: 数学表达式字符串，例如 '2 + 3 * 4' 或 'sqrt(16)'
    
    Returns:
        计算结果字符串
    """
    try:
        # 安全地计算表达式
        # 注意: eval 在生产环境中需要沙箱保护
        allowed_names = {
            k: v for k, v in math.__dict__.items() 
            if not k.startswith("__")
        }
        allowed_names["__builtins__"] = {}
        result = eval(expression, {"__builtins__": {}}, allowed_names)
        return f"计算结果: {expression} = {result}"
    except Exception as e:
        return f"计算错误: {str(e)}"


print("=== 工具1: calculator ===")
print(f"名称: {calculator.name}")
print(f"描述: {calculator.description}")
print(f"参数模式: {calculator.args_schema.schema()}")
print()

# 直接调用工具
result = calculator.invoke({"expression": "2 + 3 * 4"})
print(f"调用 calculator('2 + 3 * 4'): {result}")

result2 = calculator.invoke({"expression": "sqrt(256)"})
print(f"调用 calculator('sqrt(256)'): {result2}")

In [ ]:
# === 示例2: 使用 Pydantic BaseModel 定义结构化参数 ===

class WeatherInput(BaseModel):
    """天气查询的输入参数"""
    city: str = Field(description="城市名称，例如 '北京'、'上海'、'Tokyo'")
    unit: str = Field(
        default="celsius",
        description="温度单位: 'celsius'(摄氏度) 或 'fahrenheit'(华氏度)",
        pattern="^(celsius|fahrenheit)$"
    )
    forecast_days: Optional[int] = Field(
        default=1,
        ge=1,
        le=7,
        description="预报天数 (1-7)"
    )


@tool(args_schema=WeatherInput)
def get_weather(city: str, unit: str = "celsius", forecast_days: int = 1) -> str:
    """
    查询指定城市的天气信息。
    
    返回当前温度和天气预报。
    """
    # 模拟天气数据
    import random
    random.seed(hash(city) % 10000)
    
    temp = random.uniform(-10, 40)
    conditions = ["晴天", "多云", "小雨", "阴天", "大风", "雪"]
    condition = conditions[hash(city + str(forecast_days)) % len(conditions)]
    
    unit_str = "°C" if unit == "celsius" else "°F"
    if unit == "fahrenheit":
        temp = temp * 9/5 + 32
    
    return f"{city} 天气: {condition}, 温度 {temp:.1f}{unit_str}, 预报 {forecast_days} 天"


print("=== 工具2: get_weather (使用 Pydantic args_schema) ===")
print(f"名称: {get_weather.name}")
print(f"描述: {get_weather.description}")
print(f"参数模式字段:")
for field_name, field_info in WeatherInput.model_fields.items():
    print(f"  - {field_name}: {field_info.annotation} (默认: {field_info.default})")
print()

# 直接调用
weather_result = get_weather.invoke({"city": "北京", "unit": "celsius", "forecast_days": 3})
print(f"调用 get_weather('北京'): {weather_result}")

weather_result2 = get_weather.invoke({"city": "上海"})  # 使用默认值
print(f"调用 get_weather('上海', 默认参数): {weather_result2}")

In [ ]:
# === 示例3: return_direct 标志 ===

class SearchInput(BaseModel):
    """搜索工具的输入参数"""
    query: str = Field(description="搜索查询关键词")
    max_results: int = Field(default=5, ge=1, le=20, description="最大返回结果数")
    source: str = Field(
        default="web",
        description="搜索来源: 'web'(全网), 'news'(新闻), 'docs'(文档)"
    )


@tool(args_schema=SearchInput, return_direct=True)
def web_search(query: str, max_results: int = 5, source: str = "web") -> str:
    """
    在互联网上搜索信息。
    
    当需要查找最新信息、事实或外部知识时使用此工具。
    return_direct=True 意味着工具结果直接返回给用户，不经过 LLM 再次处理。
    """
    # 模拟搜索结果
    import hashlib
    results = []
    for i in range(min(max_results, 5)):
        hash_val = int(hashlib.md5(f"{query}{i}".encode()).hexdigest(), 16) % 1000
        results.append(f"[结果{i+1}] 关于'{query}'的{source}搜索结果 #{hash_val}: 这是模拟的搜索内容...")
    
    return "\n".join(results)


print("=== 工具3: web_search (return_direct=True) ===")
print(f"名称: {web_search.name}")
print(f"return_direct: {web_search.return_direct}")
print()

# 直接调用
search_result = web_search.invoke({"query": "LangChain 最新版本", "max_results": 3, "source": "web"})
print(f"搜索结果:\n{search_result}")
print()
print("# return_direct=True 的含义:")
print("# 当模型调用此工具时，工具的输出直接作为最终回答返回")
print("# 适用于: 搜索、计算器等不需要 LLM 二次处理的工具")

## 2. StructuredTool.from_function() - 编程式工具创建

当无法使用装饰器时（例如动态创建工具），使用 StructuredTool.from_function()。

In [ ]:
from langchain_core.tools import StructuredTool
from datetime import datetime, timedelta

# === 编程式创建工具 ===

def get_current_time(timezone: str = "Asia/Shanghai") -> str:
    """
    获取当前日期和时间。
    
    Args:
        timezone: 时区，例如 'Asia/Shanghai', 'America/New_York', 'Europe/London'
    """
    now = datetime.now()
    # 简化处理 - 生产环境应使用 pytz 库
    return f"当前时间 ({timezone}): {now.strftime('%Y-%m-%d %H:%M:%S')}"


class TimeInput(BaseModel):
    """时间查询输入"""
    timezone: str = Field(
        default="Asia/Shanghai",
        description="时区代码，例如 'Asia/Shanghai', 'America/New_York'"
    )


# 使用 StructuredTool.from_function() 创建工具
time_tool = StructuredTool.from_function(
    func=get_current_time,
    name="get_current_time",
    description="获取指定时区的当前日期和时间。用于回答'现在几点？'、'今天几号？'等问题。",
    args_schema=TimeInput,
    return_direct=False,  # LLM 可以基于结果进一步回答
    handle_tool_errors=True,  # 自动处理工具执行错误
)

print("=== 使用 StructuredTool.from_function() 创建的工具 ===")
print(f"名称: {time_tool.name}")
print(f"描述: {time_tool.description}")
print()

time_result = time_tool.invoke({"timezone": "Asia/Shanghai"})
print(f"调用结果: {time_result}")


print("\n" + "=" * 60)

# === 批量创建工具（动态场景） ===
print("批量创建多个货币转换工具:")
print("-" * 40)

exchange_rates = {
    "USD_CNY": 7.25,
    "EUR_CNY": 7.85,
    "JPY_CNY": 0.048,
    "GBP_CNY": 9.15,
}

def create_currency_tool(from_currency: str, to_currency: str, rate: float) -> StructuredTool:
    """动态创建货币转换工具"""
    
    class CurrencyInput(BaseModel):
        amount: float = Field(description=f"要转换的{from_currency}金额", gt=0)
    
    def convert(amount: float) -> str:
        result = amount * rate
        return f"{amount:.2f} {from_currency} = {result:.2f} {to_currency} (汇率: {rate})"
    
    return StructuredTool.from_function(
        func=convert,
        name=f"convert_{from_currency}_to_{to_currency}",
        description=f"将{from_currency}转换为{to_currency}，当前汇率为{rate}",
        args_schema=CurrencyInput,
    )


currency_tools = []
for pair, rate in exchange_rates.items():
    from_c, to_c = pair.split("_")
    tool = create_currency_tool(from_c, to_c, rate)
    currency_tools.append(tool)
    print(f"  创建工具: {tool.name}")

# 测试一个工具
usd_tool = currency_tools[0]
print(f"\n测试 {usd_tool.name}(100): {usd_tool.invoke({'amount': 100})}")

## 3. ToolException - 工具错误处理

ToolException 用于在工具执行失败时提供结构化错误信息。

In [ ]:
from langchain_core.tools import ToolException

# === 带自定义错误处理的工具 ===

class DatabaseInput(BaseModel):
    """数据库查询输入"""
    sql: str = Field(description="要执行的SQL查询语句")
    limit: int = Field(default=100, ge=1, le=1000, description="返回行数限制")


def handle_db_error(error: Exception) -> str:
    """自定义错误处理函数"""
    return f"数据库查询失败: {type(error).__name__} - {str(error)}。请检查SQL语法并重试。"


@tool(args_schema=DatabaseInput, handle_tool_errors=handle_db_error)
def query_database(sql: str, limit: int = 100) -> str:
    """
    执行数据库查询。
    
    支持 SELECT 查询语句。
    """
    # 模拟数据库
    forbidden_keywords = ["DROP", "DELETE", "UPDATE", "INSERT", "ALTER", "TRUNCATE"]
    sql_upper = sql.upper()
    
    for keyword in forbidden_keywords:
        if keyword in sql_upper:
            raise ToolException(f"安全限制: 不允许执行 {keyword} 操作。仅支持 SELECT 查询。")
    
    if not sql_upper.strip().startswith("SELECT"):
        raise ToolException("仅支持 SELECT 查询语句")
    
    # 模拟查询结果
    return f"查询成功: [{sql[:50]}...] 返回 {min(limit, 5)} 行结果（模拟数据）"


print("=== 工具错误处理演示 ===\n")

# 成功调用
try:
    result = query_database.invoke({
        "sql": "SELECT * FROM users WHERE age > 18 ORDER BY created_at DESC",
        "limit": 10
    })
    print(f"成功: {result}")
except ToolException as e:
    print(f"ToolException: {e}")

# 触发错误 - 使用 DROP
print()
try:
    result = query_database.invoke({
        "sql": "DROP TABLE users",
        "limit": 10
    })
except ToolException as e:
    print(f"ToolException 已捕获: {e}")

# 触发错误 - 使用 DELETE
try:
    result = query_database.invoke({
        "sql": "DELETE FROM users WHERE id=1",
        "limit": 10
    })
except ToolException as e:
    print(f"ToolException 已捕获: {e}")

print()
print("# handle_tool_errors 的三种模式:")
print("# 1. handle_tool_errors=True (默认): 返回异常字符串作为工具结果")
print("# 2. handle_tool_errors='continue': 同上，返回错误字符串")
print("# 3. handle_tool_errors=handle_db_error: 调用自定义错误处理函数")
print("# 4. handle_tool_errors=False/未设置: 异常直接向上抛出")

## 4. bind_tools() - 将工具绑定到模型

bind_tools() 将工具定义附加到 ChatOpenAI 模型，使模型能够识别并决定何时调用工具。

In [ ]:
print("=== bind_tools() - 工具绑定到模型 ===\n")

# 收集所有工具
all_tools = [calculator, get_weather, web_search, time_tool] + currency_tools[:1]

print(f"可用工具列表 ({len(all_tools)} 个):")
for t in all_tools:
    print(f"  - {t.name}: {t.description[:60]}...")

print("\n" + "=" * 60)

# 获取工具定义的 JSON Schema（用于 bind_tools）
print("# 工具定义转 JSON Schema:")
tool_schemas = [t.args_schema.schema() if t.args_schema else {} for t in all_tools[:2]]
for i, schema in enumerate(tool_schemas):
    print(f"\n  工具 {i+1} ({all_tools[i].name}) 的 schema:")
    print(f"    属性: {list(schema.get('properties', {}).keys())}")

print("\n" + "=" * 60)

print("# 绑定到模型:")
print("""
from langchain_openai import ChatOpenAI

# 创建模型实例
llm = ChatOpenAI(model="gpt-4", temperature=0)

# 绑定工具 - 模型现在知道有哪些工具可用
llm_with_tools = llm.bind_tools(
    tools=[calculator, get_weather, web_search, time_tool],
    tool_choice="auto",  # 'auto': 模型自动决定; 'none': 不使用; 'required': 必须使用
    parallel_tool_calls=True,  # 是否允许并行调用多个工具
)

# 发送消息，模型决定是否使用工具
from langchain_core.messages import HumanMessage

response = llm_with_tools.invoke([
    HumanMessage(content="北京今天天气怎么样？")
])

# 检查模型是否要求调用工具
if hasattr(response, 'tool_calls') and response.tool_calls:
    print(f"模型想要调用 {len(response.tool_calls)} 个工具:")
    for tc in response.tool_calls:
        print(f"  - 工具: {tc['name']}")
        print(f"    参数: {tc['args']}")
else:
    print(f"模型直接回答: {response.content}")
""")

print("\n# tool_choice 参数说明:")
print("# - 'auto' (默认): 模型自行决定是否调用工具")
print("# - 'none': 禁止模型调用任何工具")
print("# - 'required': 强制模型必须调用工具")
print("# - 指定工具名如 'calculator': 强制模型使用特定工具")
print()
print("# parallel_tool_calls:")
print("# - True: 允许模型在一次响应中发起多个工具调用")
print("# - False: 每次只能调用一个工具")

## 5. ToolNode (LangGraph) - 在图中执行工具调用

ToolNode 是 LangGraph 预构建的节点，负责执行模型请求的工具调用。

In [ ]:
print("=== ToolNode - LangGraph 工具执行 ===\n")

print("# LangGraph 工具调用工作流:")
print("# 1. agent 节点: LLM 分析用户消息，决定调用哪些工具")
print("# 2. tools 节点 (ToolNode): 执行工具调用，获取结果")
print("# 3. agent 节点: LLM 基于工具结果生成最终回复")
print()

print("# 完整代码:")
print("""
from langgraph.prebuilt import ToolNode, create_react_agent
from langchain_openai import ChatOpenAI

# 定义工具
@tool
def add(a: int, b: int) -> int:
    '''两个整数相加'''
    return a + b

@tool
def multiply(a: int, b: int) -> int:
    '''两个整数相乘'''
    return a * b

# 创建模型
llm = ChatOpenAI(model="gpt-4")

# 方法1: 使用 ToolNode 手动构建图
from langgraph.prebuilt import ToolNode
from langgraph.graph import StateGraph, MessagesState, START, END

tool_node = ToolNode([add, multiply])

# 定义工作流
workflow = StateGraph(MessagesState)

# 添加节点
workflow.add_node("agent", lambda state: llm_with_tools.invoke(state["messages"]))
workflow.add_node("tools", tool_node)

# 添加边
workflow.add_edge(START, "agent")
workflow.add_conditional_edges(
    "agent",
    lambda state: "tools" if state["messages"][-1].tool_calls else END,
)
workflow.add_edge("tools", "agent")

# 编译并运行
app = workflow.compile()
result = app.invoke({"messages": [("user", "3加5再乘以2等于多少？")]})
""")

print("\n# 方法2: 使用 create_react_agent 简化版（推荐）")
print("""
from langgraph.prebuilt import create_react_agent

agent = create_react_agent(
    model=llm,
    tools=[add, multiply, get_weather, web_search],
)

# 直接调用
result = agent.invoke({
    "messages": [("user", "东京今天天气如何？另外搜索一下LangChain最新版本")]
})

for msg in result["messages"]:
    if hasattr(msg, 'content') and msg.content:
        print(f"[{msg.type}]: {msg.content}")
    if hasattr(msg, 'tool_calls') and msg.tool_calls:
        print(f"[{msg.type}]: 调用工具 -> {[tc['name'] for tc in msg.tool_calls]}")
""")

print("\n# ToolNode 的核心功能:")
print("# 1. 解析 AIMessage 中的 tool_calls")
print("# 2. 查找对应的工具函数")
print("# 3. 执行工具并捕获结果")
print("# 4. 将结果封装为 ToolMessage")
print("# 5. 支持并行执行多个工具调用")
print()
print("# handle_tool_errors 在 ToolNode 中:")
print("# tool_node = ToolNode(")
print("#     tools=[...],")
print("#     handle_tool_errors=True,  # 'continue': 将错误作为消息内容返回")
print("#                                # 'raise': 抛出异常")
print("# )")

## 6. 完整的工具调用流程演示

模拟一个完整的工具调用周期：用户提问 -> 模型决策 -> 工具执行 -> 模型回答。

In [ ]:
import json
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage

print("=== 完整的工具调用流程模拟 ===\n")

# 步骤1: 用户提问
user_query = "请帮我计算 (15 + 27) * 3 的结果，同时查一下北京明天的天气"
print(f"【步骤1】用户提问: {user_query}\n")

# 步骤2: 模型分析并决定工具调用（模拟）
print("【步骤2】模型分析 -> 决定调用工具")

# 模拟模型返回的 tool_calls
simulated_tool_calls = [
    {
        "name": "calculator",
        "args": {"expression": "(15 + 27) * 3"},
        "id": "call_001"
    },
    {
        "name": "get_weather",
        "args": {"city": "北京", "unit": "celsius", "forecast_days": 2},
        "id": "call_002"
    },
]

print(f"  并行调用 {len(simulated_tool_calls)} 个工具:")
for tc in simulated_tool_calls:
    print(f"    -> {tc['name']}({json.dumps(tc['args'], ensure_ascii=False)})")

# 步骤3: 执行工具
print(f"\n【步骤3】执行工具:")

tool_results = []
for tc in simulated_tool_calls:
    tool_name = tc["name"]
    tool_args = tc["args"]
    
    if tool_name == "calculator":
        result = calculator.invoke(tool_args)
    elif tool_name == "get_weather":
        result = get_weather.invoke(tool_args)
    else:
        result = f"未知工具: {tool_name}"
    
    tool_results.append({
        "tool_call_id": tc["id"],
        "name": tool_name,
        "content": result
    })
    print(f"  {tool_name} -> {result}")

# 步骤4: 模型基于工具结果生成回答（模拟）
print(f"\n【步骤4】模型基于工具结果生成回答:")

final_answer = f"""根据计算和查询结果:

1. 计算结果: (15 + 27) * 3 = {(15 + 27) * 3}

2. 北京天气: 明天多云，温度约 25°C

希望对您有帮助！"""

print(final_answer)

print("\n" + "=" * 60)
print("工具调用流程图解:")
print("-" * 40)
print("""
  User Message
       |
       v
  [LLM 分析] ----(不需要工具)----> [最终回答]
       |
       | (需要工具)
       v
  [AIMessage with tool_calls]
       |
       v
  [ToolNode 执行工具] -----(并行执行多个)-----
       |                    |           |
       v                    v           v
  [ToolMessage1]    [ToolMessage2]  [ToolMessage3]
       |                    |           |
       +--------------------+-----------+
       |
       v
  [LLM 综合结果]
       |
       v
  [最终 AIMessage 回答]
""")

print("# 关键要点:")
print("# 1. bind_tools() 让模型知道有哪些工具可用")
print("# 2. 模型决定是否调用、调用哪个、传什么参数")
print("# 3. ToolNode 负责实际的工具执行")
print("# 4. 支持并行调用多个独立工具（如本例中计算和天气可同时进行）")
print("# 5. 工具结果以 ToolMessage 形式返回给模型")
print("# 6. 模型综合所有结果生成最终回答")

## 总结

| 概念 | 说明 | 关键参数 |
|------|------|----------|
| @tool | 装饰器方式创建工具 | args_schema, return_direct |
| StructuredTool | 编程式创建工具 | func, name, description, args_schema |
| ToolException | 工具错误处理 | handle_tool_errors |
| bind_tools() | 工具绑定到模型 | tools, tool_choice, parallel_tool_calls |
| ToolNode | LangGraph 工具执行节点 | tools, handle_tool_errors |
| create_react_agent | 快速创建工具调用代理 | model, tools |